In [1]:
%load_ext autoreload
%autoreload 2
%aimport load.loader

In [22]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from torch.utils.data import DataLoader

from load.loader import ArcadeDataset
from parse.parser import ArcadeParser

In [3]:
keys = ["C_Lav_"]

In [4]:
parser = ArcadeParser()
# parser.parse_files_to_csv(key="C_Lav_")
# parser.parse_graph_metrics_to_csv(key="C_Lav_")
# parser.parse_cell_metrics_to_csv(key="C_Lav_")

In [13]:
dataset = ArcadeDataset(keys, split_ratio=0.2)
print(f"{len(dataset)} samples in dataset")

train_subset = dataset.train_subset()
test_subset = dataset.test_subset()

print(f"{len(train_subset)} samples in train subset")
print(f"{len(test_subset)} samples in test subset")

train_loader = DataLoader(train_subset, batch_size=1, shuffle=True)
test_loader = DataLoader(test_subset, batch_size=1, shuffle=True)

3100 samples in dataset
2480 samples in train subset
620 samples in test subset


# PCA

In [45]:
train_features = []
for X, y in train_loader:
    train_features.append(X[0])

# Remove samples with inf
train_features = np.array(train_features)
train_features = train_features[~np.isinf(train_features).any(axis=1)]
print(f"{len(train_features)} samples in train features after removing inf")

num_components = 2
pca = PCA(n_components=num_components)
pca.fit(train_features)
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")

2477 samples in train features after removing inf
Explained variance ratio: [0.9446593  0.05083537]
Explained variance: [927048.65457184  49887.684966  ]
PCA components: [[-8.43385110e-03 -9.99320238e-01 -1.20827337e-07 -2.32633277e-06
  -5.29019326e-05  1.61942564e-05 -1.82020429e-04 -4.34170237e-37
  -6.52140153e-07 -2.21827406e-04  2.07790539e-02  2.92422267e-02
  -4.40976901e-04 -2.77795680e-04 -3.78857459e-04 -2.09144279e-04
   2.89468878e-06  2.89468878e-06  5.78937757e-06  1.91294027e-07
   1.41604759e-07 -5.79757868e-08  1.01399048e-07 -0.00000000e+00
  -7.14193161e-04  2.98242409e-07  2.99294051e-07 -1.79516153e-07
   5.17953033e-06  5.43878716e-06  1.06183175e-05]
 [-1.23699051e-01  3.66032697e-02 -1.42038416e-06 -2.63497615e-05
  -2.61243258e-04 -2.62929834e-05  3.16886350e-04 -3.64462015e-36
  -1.39561728e-04 -1.37981765e-04  6.15313952e-01  7.77557328e-01
  -6.80161936e-03 -2.94705524e-03 -5.38394339e-03 -2.25078679e-03
   3.05918560e-05  3.05918560e-05  6.11837119e-05 -2.

In [57]:
# Plot a test sample in the PCA space
test_features = []
test_labels = []
for X, y in test_loader:
    test_features.append(X[0])
    test_labels.append(y)

test_features = np.array(test_features)
test_labels = np.array(test_labels)
inf_indices = np.isinf(test_features).any(axis=1)
test_features = test_features[~inf_indices]
test_labels = test_labels[~inf_indices]

timepoint_labels = [label['timepoint'] for label in test_labels]

print(f"{len(test_features)} samples in test features after removing inf")

test_features_pca = pca.transform(test_features)
# Color scatter by timepoint

plt.scatter(test_features_pca[:, 0], test_features_pca[:, 1], c=timepoint_labels)
plt.show()

620 samples in test features after removing inf


IndexError: only integers, slices (`:`), ellipsis (`...`), numpy.newaxis (`None`) and integer or boolean arrays are valid indices